Settings


In [ ]:
# Path of the model ending in '_best_model_state.pth'. The models normaly are in '../models/'
MODEL_PATH = f'../models/WavLM-Base-joint_ECAPA-TDNN_Random-Triplet-Mining_BSI-Deepfake_best_model_state.pth'

# Number of audios per Batch, should be positive integer
BATCHES = 4 

# What labels to test (train|test|valid)
LABELS = "test"

# Limit of how many speaker to limit to. If it is less than 1, all speakers are used
# SET   | Min   | Max
# ------------
# train | 9     | 15
# test  | 14    | 15
# valid | 1     | 6
NUMBER_OF_SPEAKER = 1

# Limit of how many deepfakes per method per speaker to limit to. If it is less than 1, all samples are used  
# SET   | Min   | Max
# ------------
# train | 1     | 180
# test  | 1     | 180
# valid | 1     | 135
NUMBER_OF_DEEPFAKES = 1

DISTANCES = 5

Imports

In [ ]:
import torch
from models import WavLM_Base_ECAPA_TDNN
import pandas as pd
import librosa
import math
import random
import itertools
from utils.distance import compute_distance, l2_normalize
from sklearn.metrics import roc_curve
import numpy as np
from tqdm import tqdm
import plotly.express as px
from utils import load_deepfake_dataset
from dataloader import RandomTripletLossDataset, BSILoader

Functions for reading audio and extracting embeddings

In [ ]:
def read_audio_batch(filenames, max_length=0):
    waveforms = []
    for filename in filenames:
        waveform, _ = librosa.load(filename, sr=16000)  # Load audio file
        waveform, _ = librosa.effects.trim(waveform, top_db=35)  # Trim silence
        waveform = torch.tensor(waveform, dtype=torch.float32)  # Convert to torch tensor
        if max_length > 0 and waveform.shape[0] > max_length:
            # Randomly select a segment if waveform is longer than max_length
            start_sample = random.randint(0, waveform.shape[-1] - max_length)
            end_sample = start_sample + max_length
            waveform = waveform[start_sample:end_sample]
        waveforms.append(waveform)
    
    # Pad waveforms to the same length
    waveforms = torch.nn.utils.rnn.pad_sequence(waveforms, batch_first=True)
    
    return waveforms

def extract_embeddings_batch(model, waveforms):
    waveforms = waveforms.to('cuda')  # Add channel dimension and move to GPU
    with torch.no_grad():
        embeddings = model(waveforms)  # Get embeddings from the model
    return l2_normalize(embeddings.cpu())  # Normalize and move back to CPU

Function to add check distances

In [ ]:
def add_check_distances(check_data):
    speakers = check_data["speaker"].unique()
    for speaker in tqdm(speakers, desc="Speakers", total=len(speakers), position=0):
        speaker_data = check_data[check_data["speaker"] == speaker]
        for idx, deepfake in tqdm(speaker_data.iterrows(), desc="Deepfakes", position=1, leave=False):
            unique_ids = list(set([item for sublist in deepfake["positive_combinations"] for item in sublist]))
            if len(unique_ids) > DISTANCES:
                unique_ids = unique_ids[:DISTANCES]

            embeddings = []
            for utterance in unique_ids:
                embedding = data_list.loc[(data_list['utterance'] == utterance) & (data_list['is_genuine'] == 1), 'embeddings'].values[0]
                embeddings.append(torch.tensor(embedding))
            embeddings = torch.stack(embeddings)

            stacked_embeddings = torch.stack([torch.tensor(deepfake["embeddings"])] * 5)

            distances = compute_distance(embeddings, stacked_embeddings)
            data_list.at[idx, "check_distances"] = distances.tolist()
            data_list.at[idx, "check_utterances"] = unique_ids


Function to add eer, threshold and result

In [ ]:
def add_check(check, data_list):
    fpr, tpr, thresholds = roc_curve(1 - data_list['is_genuine'], data_list[check])
    fnr = 1 - tpr
    eer_index = np.nanargmin(np.absolute(fpr - fnr))
    eer = fpr[eer_index]
    eer_threshold = thresholds[eer_index]

    data_list[f"{check}_is_genuine"] = data_list[check].apply(lambda x: x <= eer_threshold)
    data_list[f"{check}_eer"] = eer
    data_list[f"{check}_threshold"] = eer_threshold

    return data_list

def add_all_checks(data_list):
    data_list = add_check("check_1", data_list)
    data_list = add_check("check_2", data_list)
    for i in range(DISTANCES):
        data_list = add_check(f"check_3_{i+1}", data_list)
    return data_list

Loading the model

In [ ]:
model = WavLM_Base_ECAPA_TDNN(frozen=False)
state = torch.load(MODEL_PATH)
model.load_state_dict(state)
model.to('cuda')
model.eval()
print("Model loaded")

Loading data

In [ ]:
train_labels, valid_labels, test_labels = load_deepfake_dataset("BSI")
if LABELS == "train":
    labels = train_labels
elif LABELS == "test":
    labels = test_labels
else:
    labels = valid_labels

dataset = RandomTripletLossDataset(loader=BSILoader(labels, lambda x: x, 0))
data_list = dataset.data_list

if NUMBER_OF_SPEAKER > 0:
    speakers = data_list['speaker'].unique()
    selected_speakers = random.sample(list(speakers), NUMBER_OF_SPEAKER)
    data_list = data_list[data_list['speaker'].isin(selected_speakers)]

if NUMBER_OF_DEEPFAKES > 0:
    grouped = data_list.groupby(['method_name', 'speaker'])
    sampled_deepfakes = []

    for _, group in grouped:
        deepfakes = group[group['is_genuine'] == 0]
        if len(deepfakes) >= NUMBER_OF_DEEPFAKES:
            sampled = deepfakes.sample(NUMBER_OF_DEEPFAKES)
            sampled_deepfakes.append(sampled)
        else:
            sampled_deepfakes.append(deepfakes)

    sampled_deepfakes_df = pd.concat(sampled_deepfakes)
    genuine_entries = data_list[data_list["is_genuine"] == 1]
    data_list = pd.concat([genuine_entries, sampled_deepfakes_df])

data_list.head()

Create all embeddings

In [ ]:
DISTANCES = 5

def chunker(data, chunk_size):
    for start in range(0, len(data), chunk_size):
        yield data.iloc[start:start + chunk_size]

total_chunks = len(data_list) // BATCHES + (1 if len(data_list) % BATCHES != 0 else 0)
all_embeddings = []
for chunk in tqdm(chunker(data_list, BATCHES), total=total_chunks):
    waveforms = read_audio_batch(chunk['filename'].tolist())

    embeddings = extract_embeddings_batch(model, waveforms)
    embeddings = embeddings.squeeze(1)  # Shape becomes [batch_size, embedding_dim]

    all_embeddings.append(embeddings.cpu().numpy())

all_embeddings = np.vstack(all_embeddings)
data_list['embeddings'] = [emb for emb in all_embeddings]

data_list.head()

Create positive distances

In [ ]:
data_list["positive_distances"] = None
data_list["positive_combinations"] = None
data_list["check_distances"] = None
data_list["check_utterances"] = None

speakers = data_list["speaker"].unique()
genuine_data = data_list[data_list['is_genuine'] == 1]

for speaker in tqdm(speakers, desc="Speakers", total=len(speakers), position=0):
    same_speakers = genuine_data[genuine_data["speaker"] == speaker]
    MAX_DISTANCES = min(math.comb(len(same_speakers), 2), DISTANCES)
    
    all_combinations = list(itertools.combinations(same_speakers.index, 2))
    selected_combinations = all_combinations[:MAX_DISTANCES]

    batch1 = []
    batch2 = []
    combinations = []
    for pair in selected_combinations:
        idx1, idx2 = pair
        embedding1 = same_speakers.loc[idx1, 'embeddings']
        embedding2 = same_speakers.loc[idx2, 'embeddings']
        utterance1 = same_speakers.loc[idx1, 'utterance']
        utterance2 = same_speakers.loc[idx2, 'utterance']
        batch1.append(torch.tensor(embedding1))
        batch2.append(torch.tensor(embedding2))
        combinations.append([utterance1, utterance2])
    
    batch1 = torch.stack(batch1)
    batch2 = torch.stack(batch2)
    
    calculated_distances = compute_distance(batch1, batch2).tolist()
    data_list.loc[data_list["speaker"] == speaker, "positive_distances"] = data_list.loc[data_list["speaker"] == speaker, "positive_distances"].apply(
        lambda x: (x if x is not None else []) + calculated_distances
    )
    data_list.loc[data_list["speaker"] == speaker, "positive_combinations"] = data_list.loc[data_list["speaker"] == speaker, "positive_combinations"].apply(
        lambda x: (x if x is not None else []) + combinations
    )

data_list.head()

Vocoder

In [ ]:
bonafide_data = data_list[data_list['method_type'] == "bonafide"]
add_check_distances(bonafide_data)

Vocoder

In [ ]:
vocoder_data = data_list[data_list['method_type'] == "Vocoder"]
add_check_distances(vocoder_data)

TTS

In [ ]:
tts_data = data_list[data_list['method_type'] == "TTS"]
add_check_distances(tts_data)

Voice Conversion

In [ ]:
vc_data = data_list[data_list['method_type'] == "VC"]
add_check_distances(vc_data)

In [ ]:
data_list.head()

Calculating deltas

- [check_1] Only 1 deepfake to original distance
- [check_2] 1 positive <> 1 check
- [check_3] Max positive vs. average 1-DISTANCES

In [ ]:
data_list["check_1"] = data_list["check_distances"].apply(lambda x: x[0])

def check_2(row):
    return row['check_distances'][0] - row['positive_distances'][0]
data_list['check_2'] = data_list.apply(check_2, axis=1)

def check_3(row, number_of_averages):
    distances = row['check_distances'][:number_of_averages]
    return (sum(distances) / number_of_averages) - max(row['positive_distances'])
for i in range(DISTANCES):
    data_list[f'check_3_{i+1}'] = data_list.apply(lambda row: check_3(row, i+1), axis=1)


data_list.head()

EER and Thresholds

In [ ]:
data_list = add_all_checks(data_list)

vocoder_data = data_list[(data_list['method_type'] == "bonafide") | (data_list['method_type'] == "Vocoder")].copy(deep=True)
vocoder_data = add_all_checks(vocoder_data)

tts_data = data_list[(data_list['method_type'] == "bonafide") | (data_list['method_type'] == "TTS")].copy(deep=True)
tts_data = add_all_checks(tts_data)

vc_data = data_list[(data_list['method_type'] == "bonafide") | (data_list['method_type'] == "TTS")].copy(deep=True)
vc_data = add_all_checks(vc_data)

Visualization

In [ ]:
viz = []
for i in range(DISTANCES):
    viz.append([
        i+1,
        data_list[f"check_3_{i+1}_eer"].iloc[0],
        "TOTAL"
    ])
    viz.append([
        i+1,
        vocoder_data[f"check_3_{i+1}_eer"].iloc[0],
        "Vocoder"
    ])
    viz.append([
        i+1,
        tts_data[f"check_3_{i+1}_eer"].iloc[0],
        "TTS"
    ])
    viz.append([
        i+1,
        vc_data[f"check_3_{i+1}_eer"].iloc[0],
        "Voice Conversion"
    ])

column_names = ['Number of checks', 'EER', 'Method Type']
df = pd.DataFrame(viz, columns=column_names)

fig = px.line(df, x='Number of checks', y='EER', color='Method Type', markers=True)
fig.show()